# Part 1: Retrieving all commits 

In [1]:
import sys
!{sys.executable} -m pip install -U python-woc pandas matplotlib

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY")

yournetid = 'mmariaru'

# WoC project id -> GitHub repo, from net2prj.csv for this netid
projects_gh = {
    'calamari-ocr_calamari':    'Calamari-OCR/calamari',
    'pdclab_peregrine':         'pdclab/peregrine',
    'juliaopt_pajarito.jl':     'JuliaOpt/Pajarito.jl',
    'mic-dkfz_batchgenerators': 'MIC-DKFZ/batchgenerators',
    'romerogroup_pyprocar':     'romerogroup/pyprocar',
    'jean-romain_lidr':         'Jean-Romain/lidR',
    'google-research_football': 'google-research/football',
    'drostlab_philentropy':     'drostlab/philentropy',
    'contextlab_hypertools':    'ContextLab/hypertools',
    'lima1_purecn':             'lima1/PureCN',
}
projects = list(projects_gh.keys())

list_df_commits = []
for prj in projects:
    prj_id = prj.lower().replace('/', '_', 2)
    commits = woc.get_values('p2c', prj_id)
    d = pd.DataFrame(commits, columns=["sha1"])
    d['project'] = prj_id
    print(prj_id, len(d), 'commits')
    list_df_commits.append(d)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
df.head(1)

calamari-ocr_calamari 851 commits
pdclab_peregrine 100 commits
juliaopt_pajarito.jl 1197 commits
mic-dkfz_batchgenerators 739 commits
romerogroup_pyprocar 1270 commits
jean-romain_lidr 4899 commits
google-research_football 347 commits
drostlab_philentropy 890 commits
contextlab_hypertools 1999 commits
lima1_purecn 1375 commits


,sha1,project
0,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,calamari-ocr_calamari


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  5%|████▉                                                                                        | 73/1367 [01:16<22:22,  1.04s/it]

Got Errors {'d9955f0a9539248fe7b6fb436f85b860caca5ddb': 'Key d9955f0a9539248fe7b6fb436f85b860caca5ddb not found in /da5_fast/All.sha1c/commit_89.tch'}


  7%|██████▌                                                                                      | 96/1367 [01:40<22:05,  1.04s/it]

Got Errors {'01308458a81e15bd2f1fe817a3b70eb153dd694b': 'Key 01308458a81e15bd2f1fe817a3b70eb153dd694b not found in /da5_fast/All.sha1c/commit_1.tch'}


  7%|██████▋                                                                                      | 99/1367 [01:43<21:56,  1.04s/it]

Got Errors {'06e82d48978bed3f090ff58ac822bb97b8449833': 'Key 06e82d48978bed3f090ff58ac822bb97b8449833 not found in /da5_fast/All.sha1c/commit_6.tch'}


... (80 outputs left out)

 30%|███████████████████████████▋                                                                | 411/1367 [07:07<16:38,  1.04s/it]

Got Errors {'f63f2488e687ff69122aab5b4036cbb25dc37759': 'Key f63f2488e687ff69122aab5b4036cbb25dc37759 not found in /da5_fast/All.sha1c/commit_118.tch'}


100%|███████████████████████████████████████████████████████████████████████████████████████████| 1367/1367 [23:34<00:00,  1.04s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,5d61ae325d2faa67e821c444d50ddf083701d138,[6e5d92ad755ca34379752c54da5326ba6ef5c6d8],Christoph Wick <wick.chr.info@gmail.com>,1536677973,+0200,Christoph Wick <wick.chr.info@gmail.com>,1536677973,+0200,added support for abbyy.xml files as input (ne...,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,calamari-ocr_calamari
1,017766a18c260883f87747dfd3ee061dd36179d7,970959e781b1aa842752600c3e27544f06d8f8b4,[1da5d4df5938ea596caeab86afea8ef822793f37],Christoph Wick <wick.chr.info@gmail.com>,1571820729,+0200,Christoph Wick <wick.chr.info@gmail.com>,1571820729,+0200,Ignoring NaN as CER in early stopping\n,017766a18c260883f87747dfd3ee061dd36179d7,calamari-ocr_calamari


In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '0123fe4d1694365ffacff2d0ae66b913acf5e0bb', 'tree': '5d61ae325d2faa67e821c444d50ddf083701d138', 'parent': ['6e5d92ad755ca34379752c54da5326ba6ef5c6d8'], 'author': 'Christoph Wick <wick.chr.info@gmail.com>', 'author_time': 1536677973, 'author_tz': '+0200', 'committer': 'Christoph Wick <wick.chr.info@gmail.com>', 'committer_time': 1536677973, 'committer_tz': '+0200', 'message': 'added support for abbyy.xml files as input (new --dataset arg to switch to ABBYY or FILE=default)\n'}


In [5]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)

In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,lima1_purecn,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,Christoph Wick <wick.chr.info@gmail.com>,1536677973,added support for abbyy.xml files as input (ne...


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid = 'mmariaru'

dfinf = pd.DataFrame({
    'project_wocid': df_commit_data['project'],
    'commit_sha1':   df_commit_data['commit'],
    'author':        df_commit_data['author'],
    'time':          df_commit_data['author_time'],
    'commit message': df_commit_data['message'],
})

# sanity check: every one of the 10 projects should have rows here
print(dfinf['project_wocid'].value_counts())

# mode='w' (overwrite) so re-running the notebook doesn't duplicate rows; header included
dfinf.to_csv(yournetid + '_project_summary.csv', index=False, sep=';', mode='w', header=True)
dfinf.head(3)

project_wocid
jean-romain_lidr            4899
contextlab_hypertools       1999
lima1_purecn                1375
romerogroup_pyprocar        1238
juliaopt_pajarito.jl        1191
drostlab_philentropy         890
calamari-ocr_calamari        850
mic-dkfz_batchgenerators     731
google-research_football     347
pdclab_peregrine             100
Name: count, dtype: int64


,project_wocid,commit_sha1,author,time,commit message
0,calamari-ocr_calamari,0123fe4d1694365ffacff2d0ae66b913acf5e0bb,Christoph Wick <wick.chr.info@gmail.com>,1536677973,added support for abbyy.xml files as input (ne...
1,calamari-ocr_calamari,017766a18c260883f87747dfd3ee061dd36179d7,Christoph Wick <wick.chr.info@gmail.com>,1571820729,Ignoring NaN as CER in early stopping\n
2,calamari-ocr_calamari,02d46bf03a7371beae462e41f6f5b6d42151566c,Christoph Wick <wick.chr.info@gmail.com>,1571995296,Added training callback to allow external prog...


# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [8]:
d = df_commit_data.copy()
woc_stats = (d.groupby('project')
               .agg(**{'#commits': ('commit', 'nunique'), '#authors': ('author', 'nunique'),
                       'From': ('author_time', 'min'), 'To': ('author_time', 'max')})
               .reset_index())
woc_stats['From_utc'] = pd.to_datetime(woc_stats['From'], unit='s', utc=True)   # readable, for the report text
woc_stats['To_utc']   = pd.to_datetime(woc_stats['To'], unit='s', utc=True)
woc_stats

,project,#commits,#authors,From,To,From_utc,To_utc
0,calamari-ocr_calamari,850,48,1521559350,1749469622,2018-03-20 15:22:30+00:00,2025-06-09 11:47:02+00:00
1,contextlab_hypertools,1999,43,1469027405,1769667130,2016-07-20 15:10:05+00:00,2026-01-29 06:12:10+00:00
2,drostlab_philentropy,890,11,1424096347,1762174389,2015-02-16 14:19:07+00:00,2025-11-03 12:53:09+00:00
3,google-research_football,347,78,1559318565,1769902631,2019-05-31 16:02:45+00:00,2026-01-31 23:37:11+00:00
4,jean-romain_lidr,4899,36,1455710102,1759947297,2016-02-17 11:55:02+00:00,2025-10-08 18:14:57+00:00
5,juliaopt_pajarito.jl,1191,17,1456860843,1581804086,2016-03-01 19:34:03+00:00,2020-02-15 22:01:26+00:00
6,lima1_purecn,1375,28,1460665299,1771532264,2016-04-14 20:21:39+00:00,2026-02-19 20:17:44+00:00
7,mic-dkfz_batchgenerators,731,59,1489747027,1758661078,2017-03-17 10:37:07+00:00,2025-09-23 20:57:58+00:00
8,pdclab_peregrine,100,6,1586109630,1741800437,2020-04-05 18:00:30+00:00,2025-03-12 17:27:17+00:00
9,romerogroup_pyprocar,1238,37,1521992052,1762090662,2018-03-25 15:34:12+00:00,2025-11-02 13:37:42+00:00


In [9]:
import httpx, time

headers = {'Accept': 'application/vnd.github+json'}

rows = []
with httpx.Client(follow_redirects=True, timeout=30, headers=headers) as c:
    for wid, repo in projects_gh.items():
        row = {'project': wid, 'repo': repo, '#stars': None, '#forks': None, 'LastGHCommitDate': None}

        info = c.get(f'https://api.github.com/repos/{repo}').json()
        if 'message' in info:
            print(repo, '-> repo info error:', info['message'])
        else:
            row['#stars'] = info.get('stargazers_count')
            row['#forks'] = info.get('forks_count')

        last = c.get(f'https://api.github.com/repos/{repo}/commits', params={'per_page': 1}).json()
        if isinstance(last, list) and last:
            row['LastGHCommitDate'] = last[0]['commit']['committer']['date'][:10].replace('-', '/')
        elif isinstance(last, dict) and 'message' in last:
            print(repo, '-> commits error:', last['message'])

        rows.append(row)  # always append, even on partial failure, so the column always exists
        time.sleep(0.2)

gh_stats = pd.DataFrame(rows)
gh_stats.to_csv('github_stats.csv', index=False)
gh_stats

,project,repo,#stars,#forks,LastGHCommitDate
0,calamari-ocr_calamari,Calamari-OCR/calamari,1198,209,2026/05/13
1,pdclab_peregrine,pdclab/peregrine,208,38,2025/03/12
2,juliaopt_pajarito.jl,JuliaOpt/Pajarito.jl,141,21,2026/09/17
3,mic-dkfz_batchgenerators,MIC-DKFZ/batchgenerators,1175,230,2026/10/08
4,romerogroup_pyprocar,romerogroup/pyprocar,224,87,2025/09/20
5,jean-romain_lidr,Jean-Romain/lidR,713,141,2026/09/21
6,google-research_football,google-research/football,3667,1386,2025/06/17
7,drostlab_philentropy,drostlab/philentropy,150,22,2025/12/14
8,contextlab_hypertools,ContextLab/hypertools,1891,164,2026/10/09
9,lima1_purecn,lima1/PureCN,151,33,2026/10/06


# Part 3.1: Examining commits before and after the gap

In [10]:
GAP_START = 'LongestGapStart (YYYY-MM), '   # header text copied verbatim from the example
GAP_END   = 'LongestGapEnd (YYYY-MM),'

# Part 3 builds on files written earlier:
#   mmariaru_project_summary.csv  <- Part 1 (this notebook)
#   mmariaru_project_stats.csv    <- Part 2 (mmariaru_vis.ipynb): gap columns and pattern
summary = pd.read_csv(yournetid + '_project_summary.csv', sep=';')
summary['dt'] = pd.to_datetime(summary['time'], unit='s', utc=True)

gap_stats = pd.read_csv(yournetid + '_project_stats.csv', sep=';')[
    ['project', GAP_START, GAP_END, 'LongestGapLength',
     'NumCommitsAfterLastGap', 'pattern', 'NumberOfGapsInTimeline']]
gap_stats

,project,"LongestGapStart (YYYY-MM),","LongestGapEnd (YYYY-MM),",LongestGapLength,NumCommitsAfterLastGap,pattern,NumberOfGapsInTimeline
0,calamari-ocr_calamari,2023-02,2023-07,6,110,declining,2
1,contextlab_hypertools,2022-09,2024-02,18,51,declining,7
2,drostlab_philentropy,2016-06,2017-03,10,391,irregular,13
3,google-research_football,2022-10,2023-07,10,45,declining,4
4,jean-romain_lidr,2023-06,2023-07,2,141,declining,0
5,juliaopt_pajarito.jl,2019-07,2020-01,7,13,declining,2
6,lima1_purecn,2022-05,2022-09,5,146,declining,5
7,mic-dkfz_batchgenerators,2022-06,2022-09,4,122,declining,3
8,pdclab_peregrine,2023-10,2025-02,17,4,declining,5
9,romerogroup_pyprocar,2022-09,2022-12,4,753,rising,1


In [11]:
THEME_RULES = [
    # (theme, keywords checked against the lowercased commit message)
    ('Security patches', ['cve', 'security', 'vulnerab', 'exploit']),
    ('Automated bot contributions', []),  # checked against author instead, see below
    ('Release', ['release', 'changelog', 'bump version', 'v{num}'.format(num=''), 'tag ']),
    ('Dependency updates', ['bump', 'dependenc', 'requirements', 'upgrade', 'deps']),
    ('Documentation updates', ['readme', 'docs', 'documentation', 'typo', 'comment']),
    ('Bug fixes', ['fix', 'bug', 'error', 'crash', 'issue #']),
    ('Feature development', ['add', 'implement', 'feature', 'support for', 'new ']),
]
BOT_AUTHOR_HINTS = ['bot', 'dependabot', 'github-actions', 'renovate']

def classify_theme(author, message):
    a = str(author).lower()
    if any(h in a for h in BOT_AUTHOR_HINTS):
        return 'Automated bot contributions'
    m = str(message).lower()
    for theme, kws in THEME_RULES:
        if theme == 'Automated bot contributions':
            continue
        if any(kw in m for kw in kws):
            return theme
    return 'Other'

def top2_themes(msgs_authors):
    from collections import Counter
    cnt = Counter(classify_theme(a, m) for a, m in msgs_authors)
    top = [t for t, _ in cnt.most_common(2)]
    while len(top) < 2:
        top.append('Other')
    return top

print('Heuristic keyword classifier ready — spot-check a sample before trusting it:')
sample = summary.sample(min(5, len(summary)))[['author', 'commit message']]
for _, r in sample.iterrows():
    print(classify_theme(r['author'], r['commit message']), '|', str(r['commit message'])[:70].replace(chr(10), ' '))

Heuristic keyword classifier ready — spot-check a sample before trusting it:
Other | update cluster example 
Release | Merge branch 'master' into devel 
Bug fixes | fixing a bug which tested transposed matrix rows rather than transpose
Other | also in wrapper 
Documentation updates | update README 


In [12]:
gap_commit_rows = []
theme_rows = []

for prj in projects:
    sub = summary[summary['project_wocid'] == prj].sort_values('dt').reset_index(drop=True)
    g = gap_stats[gap_stats['project'] == prj].iloc[0]
    if pd.isna(g[GAP_START]):
        theme_rows.append({'project': prj, 'BeforeThemes': 'N/A', 'AfterThemes': 'N/A'})
        continue

    gap_start_ts = pd.Period(g[GAP_START], freq='M').start_time.tz_localize('UTC')
    gap_end_ts = pd.Period(g[GAP_END], freq='M').end_time.tz_localize('UTC')

    before = sub[sub['dt'] < gap_start_ts].tail(10)
    after = sub[sub['dt'] > gap_end_ts].head(10)

    for _, r in before.iterrows():
        gap_commit_rows.append({'pre_post': 'pre', 'project_wocid': prj, 'commit_sha1': r['commit_sha1'],
                                 'author': r['author'], 'time': r['time'], 'commit message': r['commit message']})
    for _, r in after.iterrows():
        gap_commit_rows.append({'pre_post': 'post', 'project_wocid': prj, 'commit_sha1': r['commit_sha1'],
                                 'author': r['author'], 'time': r['time'], 'commit message': r['commit message']})

    bt = top2_themes(zip(before['author'], before['commit message']))
    at = top2_themes(zip(after['author'], after['commit message']))
    theme_rows.append({'project': prj, 'BeforeThemes': ':'.join(bt), 'AfterThemes': ':'.join(at)})

gap_commits_df = pd.DataFrame(gap_commit_rows)
gap_commits_df.to_csv(yournetid + '_project_gap_commits.csv', index=False, sep=';')
theme_stats = pd.DataFrame(theme_rows)
theme_stats

,project,BeforeThemes,AfterThemes
0,calamari-ocr_calamari,Release:Bug fixes,Release:Other
1,pdclab_peregrine,Release:Feature development,Release:Other
2,juliaopt_pajarito.jl,Other:Release,Release:Other
3,mic-dkfz_batchgenerators,Bug fixes:Other,Other:Release
4,romerogroup_pyprocar,Other:Documentation updates,Other:Feature development
5,jean-romain_lidr,Other:Bug fixes,Bug fixes:Other
6,google-research_football,Feature development:Bug fixes,Other:Release
7,drostlab_philentropy,Documentation updates:Release,Other:Documentation updates
8,contextlab_hypertools,Other:Dependency updates,Release:Other
9,lima1_purecn,Release:Other,Release:Other


# Part 3.2: Interpreting the gap

In [13]:
from datetime import datetime, timezone

CUTOFF = datetime(2026, 4, 8, tzinfo=timezone.utc)
gh_stats['Currentstatus'] = gh_stats['LastGHCommitDate'].apply(
    lambda d: 'Active' if pd.notna(d) and datetime.strptime(d, '%Y/%m/%d').replace(tzinfo=timezone.utc) >= CUTOFF else 'Inactive'
)
gh_stats[['project', 'LastGHCommitDate', 'Currentstatus']]

,project,LastGHCommitDate,Currentstatus
0,calamari-ocr_calamari,2026/05/13,Active
1,pdclab_peregrine,2025/03/12,Inactive
2,juliaopt_pajarito.jl,2026/09/17,Active
3,mic-dkfz_batchgenerators,2026/10/08,Active
4,romerogroup_pyprocar,2025/09/20,Inactive
5,jean-romain_lidr,2026/09/21,Active
6,google-research_football,2025/06/17,Inactive
7,drostlab_philentropy,2025/12/14,Inactive
8,contextlab_hypertools,2026/10/09,Active
9,lima1_purecn,2026/10/06,Active


In [14]:
# RecentThemes: top-2 themes of the last 10 commits before the most recent commit date, per project
recent_rows = []
for prj in projects:
    sub = summary[summary['project_wocid'] == prj].sort_values('dt')
    recent = sub.tail(10)
    rt = top2_themes(zip(recent['author'], recent['commit message']))
    recent_rows.append({'project': prj, 'RecentThemes': ':'.join(rt)})
recent_stats = pd.DataFrame(recent_rows)
recent_stats

,project,RecentThemes
0,calamari-ocr_calamari,Release:Other
1,pdclab_peregrine,Release:Other
2,juliaopt_pajarito.jl,Release:Feature development
3,mic-dkfz_batchgenerators,Feature development:Other
4,romerogroup_pyprocar,Release:Security patches
5,jean-romain_lidr,Release:Bug fixes
6,google-research_football,Other:Automated bot contributions
7,drostlab_philentropy,Release:Documentation updates
8,contextlab_hypertools,Release:Documentation updates
9,lima1_purecn,Release:Other


In [15]:
def nearby_issues(repo, start_iso, end_iso, client, pad_days=60):
    q = f'repo:{repo} updated:{start_iso}..{end_iso}'
    r = client.get('https://api.github.com/search/issues', params={'q': q, 'per_page': 10})
    data = r.json()
    return data.get('items', [])

import pandas as pd
with httpx.Client(follow_redirects=True, timeout=30, headers=headers) as c:
    for prj, repo in projects_gh.items():
        g = gap_stats[gap_stats['project'] == prj].iloc[0]
        if pd.isna(g[GAP_START]):
            continue
        start = pd.Period(g[GAP_START], freq='M').start_time.date().isoformat()
        end = pd.Period(g[GAP_END], freq='M').end_time.date().isoformat()
        items = nearby_issues(repo, start, end, c)
        print(f'--- {prj} ({repo}), gap {start} to {end}: {len(items)} issues/PRs found ---')
        for it in items[:5]:
            print(' ', it.get('title', '')[:80], '-', it.get('html_url'))
        time.sleep(7)

--- calamari-ocr_calamari (Calamari-OCR/calamari), gap 2023-02-01 to 2023-07-31: 4 issues/PRs found ---
  Attention layer  - https://github.com/Calamari-OCR/calamari/issues/339
  What is the accuracy on Chinese/Japanese text? - https://github.com/Calamari-OCR/calamari/issues/338
   W tensorflow/core/kernels/data/generator_dataset_op.cc:103] Error occurred when - https://github.com/Calamari-OCR/calamari/issues/164
  License - https://github.com/Calamari-OCR/calamari/issues/3
--- pdclab_peregrine (pdclab/peregrine), gap 2023-10-01 to 2025-02-28: 5 issues/PRs found ---
  Issue with make command  - https://github.com/pdclab/peregrine/issues/25
  issue about graph6 - https://github.com/pdclab/peregrine/issues/24
  A question about subgraph match - https://github.com/pdclab/peregrine/issues/23
  Extending Multi Label Functionality - https://github.com/pdclab/peregrine/issues/22
  Add Startup Script - https://github.com/pdclab/peregrine/pull/21
--- juliaopt_pajarito.jl (JuliaOpt/Pajarito.jl),

In [21]:
manual_part4 = {
    'calamari-ocr_calamari': {
        'HypothesizedGapReason': 'Original developer left in 2021. Part-time maintainers only worked when something broke',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Dependency breakage required compatibility fixes and new releases',
        'WhoRecovered': 'Same long-time maintainers',
        'Notes': 'Declining pattern with 2 gaps (longest about 6 months). Activity comes in release-driven sprints and the project is still active.'
    },

    'pdclab_peregrine': {
        'HypothesizedGapReason': 'Lead developer moved on after the research publication.',
        'HasRecovered': 'No',
        'WhyRecovered': 'N/A',
        'WhoRecovered': 'N/A',
        'Notes': 'Declining pattern with 5 gaps (longest about 17 months). Only a brief outside contribution in 2025, so the project is essentially dormant.'
    },

    'juliaopt_pajarito.jl': {
        'HypothesizedGapReason': 'Core algorithm stabilized. Only light upkeep while maintainers focused elsewhere',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Language ecosystem packaging and CI changes required infrastructure updates',
        'WhoRecovered': 'Same maintainers',
        'Notes': 'Declining pattern with 2 gaps (longest about 7 months). WoC data ends in early 2020, likely because the repository later moved organizations.'
    },

    'mic-dkfz_batchgenerators': {
        'HypothesizedGapReason': 'Mature library with one lead developer; quiet periods between releases',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Lead maintainer returned for release and maintenance work',
        'WhoRecovered': 'Same lead maintainer plus occasional outside contributors',
        'Notes': 'Declining pattern with 3 short gaps (longest about 4 months). The project is still active at a low intensity.'
    },

    'romerogroup_pyprocar': {
        'HypothesizedGapReason': 'Core developer paused before a major restructuring',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Planned large refactor with new features and documentation',
        'WhoRecovered': 'Same core developer. Team grew later',
        'Notes': 'Rising pattern with only 1 gap (about 4 months). Activity kept growing and peaked in 2025.'
    },

    'jean-romain_lidr': {
        'HypothesizedGapReason': 'N/A active Project',
        'HasRecovered': 'N/A',
        'WhyRecovered': 'N/A',
        'WhoRecovered': 'N/A',
        'Notes': 'Declining pattern with no gap of 3+ months (longest 2 months). A single maintainer keeps up steady maintenance and the project is active.'
    },

    'google-research_football': {
        'HypothesizedGapReason': 'Core team stopped active development after the research release. Only outside contributions remained',
        'HasRecovered': 'Partially',
        'WhyRecovered': 'Sporadic outside contributions, then new feature and modernization work in 2025-2026',
        'WhoRecovered': 'Mostly new outside contributors',
        'Notes': 'Declining pattern with 4 gaps (longest about 10 months). Activity comes in bursts driven by outside contributors.'
    },

    'drostlab_philentropy': {
        'HypothesizedGapReason': 'Single maintainer working in bursts with long quiet periods between releases',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Maintainer returned to restructure code and prepare releases',
        'WhoRecovered': 'Same maintainer. A few outside contributors later',
        'Notes': 'Irregular pattern with 13 gaps (longest about 10 months). The project is still occasionally active.'
    },

    'contextlab_hypertools': {
        'HypothesizedGapReason': 'Lab developers moved on after the 2022 plotting refactor',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Notebook and plotting ecosystem changes broke functionality and needed compatibility updates',
        'WhoRecovered': 'Same lab members plus occasional outside contributors',
        'Notes': 'Declining pattern with 7 gaps (longest about 18 months). Maintenance now comes in sporadic bursts, the latest in early 2026.'
    },

    'lima1_purecn': {
        'HypothesizedGapReason': 'Single primary maintainer. Activity follows the host ecosystem release cycle',
        'HasRecovered': 'Yes',
        'WhyRecovered': 'Next release cycle began, prompting housekeeping and updates',
        'WhoRecovered': 'Same maintainer plus release-process contributors',
        'Notes': 'Declining pattern with 5 gaps (longest about 5 months). Activity comes in release-cycle bursts and the project is still active in 2026.'
    },
}

manual_part4_df = pd.DataFrame(manual_part4).T.reset_index().rename(columns={'index': 'project'})

stats_file = yournetid + '_project_stats.csv'
part34 = ['BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered', 'WhyRecovered',
          'WhoRecovered', 'Currentstatus', 'RecentThemes', 'Notes']

base = pd.read_csv(stats_file, sep=';')
base = base.drop(columns=[c for c in part34 if c in base.columns])

final = (base.merge(theme_stats, on='project', how='left')
             .merge(gh_stats[['project', 'Currentstatus']], on='project', how='left')
             .merge(recent_stats, on='project', how='left')
             .merge(manual_part4_df, on='project', how='left'))
final = final[list(base.columns) + part34]

final.to_csv(stats_file, sep=';', index=False)
final

,project,#commits,#authors,From,To,#stars,#forks,LastGHCommitDate,"LongestGapStart (YYYY-MM),","LongestGapEnd (YYYY-MM),",...,NumberOfGapsInTimeline,BeforeThemes,AfterThemes,HypothesizedGapReason,HasRecovered,WhyRecovered,WhoRecovered,Currentstatus,RecentThemes,Notes
0,calamari-ocr_calamari,850,48,1521559350,1749469622,1198,209,2026/05/13,2023-02,2023-07,...,2,Release:Bug fixes,Release:Other,Original developer left in 2021. Part-time mai...,Yes,Dependency breakage required compatibility fix...,Same long-time maintainers,Active,Release:Other,Declining pattern with 2 gaps (longest about 6...
1,contextlab_hypertools,1999,43,1469027405,1769667130,1891,164,2026/10/09,2022-09,2024-02,...,7,Other:Dependency updates,Release:Other,Lab developers moved on after the 2022 plottin...,Yes,Notebook and plotting ecosystem changes broke ...,Same lab members plus occasional outside contr...,Active,Release:Documentation updates,Declining pattern with 7 gaps (longest about 1...
2,drostlab_philentropy,890,11,1424096347,1762174389,150,22,2025/12/14,2016-06,2017-03,...,13,Documentation updates:Release,Other:Documentation updates,Single maintainer working in bursts with long ...,Yes,Maintainer returned to restructure code and pr...,Same maintainer. A few outside contributors later,Inactive,Release:Documentation updates,Irregular pattern with 13 gaps (longest about ...
3,google-research_football,347,78,1559318565,1769902631,3667,1386,2025/06/17,2022-10,2023-07,...,4,Feature development:Bug fixes,Other:Release,Core team stopped active development after the...,Partially,"Sporadic outside contributions, then new featu...",Mostly new outside contributors,Inactive,Other:Automated bot contributions,Declining pattern with 4 gaps (longest about 1...
4,jean-romain_lidr,4899,36,1455710102,1759947297,713,141,2026/09/21,2023-06,2023-07,...,0,Other:Bug fixes,Bug fixes:Other,N/A active Project,N/A,N/A,N/A,Active,Release:Bug fixes,Declining pattern with no gap of 3+ months (lo...
5,juliaopt_pajarito.jl,1191,17,1456860843,1581804086,141,21,2026/09/17,2019-07,2020-01,...,2,Other:Release,Release:Other,Core algorithm stabilized. Only light upkeep w...,Yes,Language ecosystem packaging and CI changes re...,Same maintainers,Active,Release:Feature development,Declining pattern with 2 gaps (longest about 7...
6,lima1_purecn,1375,28,1460665299,1771532264,151,33,2026/10/06,2022-05,2022-09,...,5,Release:Other,Release:Other,Single primary maintainer. Activity follows th...,Yes,"Next release cycle began, prompting housekeepi...",Same maintainer plus release-process contributors,Active,Release:Other,Declining pattern with 5 gaps (longest about 5...
7,mic-dkfz_batchgenerators,731,59,1489747027,1758661078,1175,230,2026/10/08,2022-06,2022-09,...,3,Bug fixes:Other,Other:Release,Mature library with one lead developer; quiet ...,Yes,Lead maintainer returned for release and maint...,Same lead maintainer plus occasional outside c...,Active,Feature development:Other,Declining pattern with 3 short gaps (longest a...
8,pdclab_peregrine,100,6,1586109630,1741800437,208,38,2025/03/12,2023-10,2025-02,...,5,Release:Feature development,Release:Other,Lead developer moved on after the research pub...,No,N/A,N/A,Inactive,Release:Other,Declining pattern with 5 gaps (longest about 1...
9,romerogroup_pyprocar,1238,37,1521992052,1762090662,224,87,2025/09/20,2022-09,2022-12,...,1,Other:Documentation updates,Other:Feature development,Core developer paused before a major restructu...,Yes,Planned large refactor with new features and d...,Same core developer. Team grew later,Inactive,Release:Security patches,Rising pattern with only 1 gap (about 4 months...
